## Super-Resolved Image Generation

The best-performing RCAN model is used to generate super-resolved images from the original maize leaf images. The same trained model is applied independently to the training, validation, and test sets while preserving the original class structure.

In [ ]:
import os
from pathlib import Path
import types

import numpy as np
import torch

from PIL import Image
from tqdm import tqdm

from model.rcan import Net


# =====================================================
# CONFIGURATION
# =====================================================
SCALE = 2

MODEL_PATH = Path("path/to/best_rcan_checkpoint.pth")

INPUT_ROOT = Path("path/to/dataset")

OUTPUT_ROOT = Path("path/to/super_resolved_images")

SUBSETS = ["train", "val", "test"]
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png"}

MODEL_CONFIG = types.SimpleNamespace(
    scale=SCALE,
    num_groups=10,
    num_blocks=20,
    num_channels=64,
    reduction=16,
    res_scale=1.0
)


# =====================================================
# DEVICE & MODEL
# =====================================================
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = Net(MODEL_CONFIG).to(device)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=False
)

# Support both full checkpoints and direct state_dict files
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    print(
        f"Loaded checkpoint from epoch "
        f"{checkpoint.get('epoch', 'N/A')}"
    )

    if "best_val_psnr" in checkpoint:
        print(
            f"Best validation PSNR: "
            f"{checkpoint['best_val_psnr']:.4f} dB"
        )
else:
    model.load_state_dict(checkpoint)

model.eval()

print(f"Device: {device}")
print(f"RCAN scale: ×{SCALE}")


# =====================================================
# SUPER-RESOLUTION FUNCTION
# =====================================================
def super_resolve(
    image_path,
    model,
    device,
    scale
):
    image = Image.open(
        image_path
    ).convert("RGB")

    image_array = np.array(
        image
    ).astype(np.float32)

    original_height, original_width = (
        image_array.shape[:2]
    )

    # Pad image so that its dimensions are divisible by scale
    pad_height = (
        scale - original_height % scale
    ) % scale

    pad_width = (
        scale - original_width % scale
    ) % scale

    if pad_height > 0 or pad_width > 0:
        image_array = np.pad(
            image_array,
            (
                (0, pad_height),
                (0, pad_width),
                (0, 0)
            ),
            mode="reflect"
        )

    # Convert image to tensor
    image_tensor = (
        torch.from_numpy(image_array)
        .permute(2, 0, 1)
        .unsqueeze(0)
        .to(device)
    )

    with torch.no_grad():
        super_resolved = model(
            image_tensor
        )

        super_resolved = torch.clamp(
            super_resolved,
            0.0,
            255.0
        )

    # Remove padding after super-resolution
    output_height = original_height * scale
    output_width = original_width * scale

    super_resolved = super_resolved[
        :,
        :,
        :output_height,
        :output_width
    ]

    # Convert tensor back to image
    output_array = (
        super_resolved
        .squeeze(0)
        .permute(1, 2, 0)
        .cpu()
        .numpy()
        .astype(np.uint8)
    )

    return Image.fromarray(
        output_array
    )


# =====================================================
# GENERATE SUPER-RESOLVED IMAGES
# =====================================================
total_processed = 0
total_failed = 0

for subset in SUBSETS:

    print(
        f"\nProcessing {subset.upper()}..."
    )

    for class_dir in sorted(
        (INPUT_ROOT / subset).iterdir()
    ):

        if not class_dir.is_dir():
            continue

        output_class_dir = (
            OUTPUT_ROOT
            / subset
            / class_dir.name
        )

        output_class_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        image_files = [
            image_path
            for image_path in sorted(
                class_dir.iterdir()
            )
            if image_path.suffix.lower()
            in VALID_EXTENSIONS
        ]

        for image_path in tqdm(
            image_files,
            desc=f"{class_dir.name:<12}"
        ):

            output_path = (
                output_class_dir
                / f"{image_path.stem}.png"
            )

            try:
                super_resolved_image = (
                    super_resolve(
                        image_path,
                        model,
                        device,
                        SCALE
                    )
                )

                super_resolved_image.save(
                    output_path
                )

                total_processed += 1

            except Exception as error:
                print(
                    f"Failed: "
                    f"{image_path.name} | "
                    f"{error}"
                )

                total_failed += 1


# =====================================================
# VERIFY OUTPUT
# =====================================================
print("\n" + "=" * 50)
print("SUPER-RESOLUTION GENERATION SUMMARY")
print("=" * 50)

for subset in SUBSETS:

    subset_dir = (
        OUTPUT_ROOT / subset
    )

    print(f"\n{subset.upper()}/")

    if not subset_dir.exists():
        print("  No output generated.")
        continue

    for class_dir in sorted(
        subset_dir.iterdir()
    ):

        if not class_dir.is_dir():
            continue

        image_count = len([
            path
            for path in class_dir.iterdir()
            if path.suffix.lower()
            in {".png", ".jpg", ".jpeg"}
        ])

        print(
            f"  {class_dir.name:<12} "
            f"→ {image_count:,} images"
        )

print(
    f"\nTotal processed : "
    f"{total_processed:,}"
)

print(
    f"Total failed    : "
    f"{total_failed:,}"
)

print(
    f"Output directory: "
    f"{OUTPUT_ROOT}"
)